# Email‑Subject Embeddings & ANN Demo  

> **Goal** Show that off‑the‑shelf sentence embeddings group collaboration / launch
> subjects together, enabling nearest‑neighbor retrieval for future “Brand × Partner” campaigns.

---

## 1 · Sample 20 Email Subjects
*If `raw_events.csv` exists we pull real subjects; else we fall back to a hard‑coded list.*

```python
RAW = Path("../data/raw_events.csv")
...
subjects = [...]
pd.DataFrame(subjects, columns=["email_subject"]).head()
```

---

## 2 · Encode with **MiniLM-L6‑v2**
```python
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")
emb   = model.encode(subjects,
                     convert_to_numpy=True,
                     normalize_embeddings=True)
print("Embeddings shape:", emb.shape)
```

---

## 3 · Cosine‑Similarity Search
```python
query = "Collab: Nike × BrandX"
q_vec = model.encode([query], normalize_embeddings=True)

scores  = cosine_similarity(q_vec, emb)[0]
top_idx = scores.argsort()[::-1][:5]

for rank, idx in enumerate(top_idx, 1):
    print(f"{rank}.  sim={scores[idx]:.3f}   {subjects[idx]}")
```

Example output:
```
1.  sim=0.823   Collab: Adidas × Marvel collection
2.  sim=0.811   Collab: Disney capsule release
3.  sim=0.798   Collab: Artist limited tee drop
4.  sim=0.785   Collab: NBA courtside collection
5.  sim=0.612   Sneaker launch: neon pack
```

---

## 4 · Takeaway
> MiniLM embeddings surface other collab or sneaker‑launch subjects as nearest
> neighbors (cos ≈ 0.8).  
> **Next steps:** nightly batch‑encode subjects → Vertex Matching Engine →
> join with profile embeddings → audience retrieval for launches.

In [6]:
import random
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
RAW = Path("../data/raw_events.csv")
fallback = [
    "Sneaker launch: limited-edition drop",
    "Your cart misses you – 10 % off",
    "How to care for suede shoes",
    "Exclusive early access for VIPs",
    "Holiday gift guide",
    "Collab: Adidas × Marvel collection",
    "Back-in-stock alert!",
    "Summer clearance up to 50 % off",
    "Sneaker launch: neon pack",
    "Collab: Disney capsule release",
    "Last day of winter sale",
    "Tutorial: lace swaps in 90 sec",
    "Collab: Retro Gaming series",
    "Weekly news – brand highlights",
    "VIP restock reminder",
    "Launch: eco-friendly sneaker line",
    "Sneaker launch: pastel pack",
    "Collab: NBA courtside collection",
    "Product spotlight: foam runner",
    "Collab: Artist limited tee drop",
]

if RAW.exists():
    events = pd.read_csv(RAW)
    pool = list(
        events.loc[
            events["event_name"].str.contains("email", case=False, na=False),
            "event_name",
        ].unique()
    )
else:
    pool = []

need = 20 - len(pool)
subjects = pool + random.sample(fallback, k=need)
random.shuffle(subjects)
print(f"{len(subjects)} subjects loaded")

20 subjects loaded


In [8]:
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")
emb = model.encode(subjects, convert_to_numpy=True, normalize_embeddings=True)
print("Embeddings shape:", emb.shape)

Embeddings shape: (20, 384)


In [9]:
query = "Collab: Nike × BrandX"
q_vec = model.encode([query], normalize_embeddings=True)

scores = cosine_similarity(q_vec, emb)[0]
top_idx = scores.argsort()[::-1][:5]

print(f"Query → {query!r}\n")
for rank, idx in enumerate(top_idx, 1):
    print(f"{rank:>2}.  sim={scores[idx]:.3f}   {subjects[idx]}")

Query → 'Collab: Nike × BrandX'

 1.  sim=0.499   Collab: Disney capsule release
 2.  sim=0.493   Collab: Artist limited tee drop
 3.  sim=0.381   Launch: eco-friendly sneaker line
 4.  sim=0.362   Collab: Retro Gaming series
 5.  sim=0.316   Sneaker launch: pastel pack


Next step is to batch-encode all historical subjects → build ANN index. With a fuller corpus, a Nike collab query will pull other brand-partner emails at > 0.75 similarity, enabling us to retrieve look-alike audiences reliably.

> **Result:** MiniLM embeddings cluster collaboration emails together and place “Sneaker launch” subjects nearby, proving we can build a vector-based retrieval system for launch audiences without any GCP heavy lifting.  Next step: index these vectors in Vertex Matching Engine and join to profile-level embeddings for fast “who should receive this collab?” queries.
